# 🧬 LLPS Protein Data Explorer - Step-by-Step Analysis

This notebook provides a transparent, step-by-step view of all data transformations applied to the LLPS protein dataset. Each cell shows exactly what is happening to your data at each stage.

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Initial Data Inspection](#2-initial-data-inspection)
3. [Location Parsing Transformations](#3-location-parsing-transformations)
4. [Data Filtering Examples](#4-data-filtering-examples)
5. [Statistical Analysis](#5-statistical-analysis)
6. [Visualizations](#6-visualizations)

---
## 1. Setup & Data Loading

First, we import all necessary libraries and load the data.

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")

In [ ]:
# Load the sample data
data_path = Path("data/sample_data.xlsx")

if data_path.exists():
    df_raw = pd.read_excel(data_path, engine='openpyxl')
    print(f"✅ Data loaded successfully!")
    print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
else:
    print("❌ Sample data not found. Please ensure data/sample_data.xlsx exists.")
    print("   You can also upload your own XLSX file and update the path above.")

---
## 2. Initial Data Inspection

Let's examine the raw data before any transformations.

In [ ]:
# View all column names
print("📋 Column names in the dataset:")
print("=" * 50)
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:2}. {col}")

In [ ]:
# Data types and non-null counts
print("📊 Data Types and Non-Null Counts:")
print("=" * 50)
df_raw.info()

In [ ]:
# Preview the first few rows
print("👀 First 5 rows of raw data:")
df_raw.head()

In [ ]:
# Basic statistics for numeric columns
print("📈 Statistical summary of numeric columns:")
df_raw.describe()

In [ ]:
# Check for missing values
print("🔍 Missing values per column:")
print("=" * 50)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))
print(f"\nTotal rows: {len(df_raw)}")

---
## 3. Location Parsing Transformations

The dashboard parses subcellular location strings into standardized categories. Let's see this step-by-step.

In [ ]:
import re

# No predefined categories needed - we extract location tags directly
# from the UniProt annotation, removing only the evidence codes in curly brackets
print("ℹ️ Location parsing approach:")
print("   1. Remove curly bracket content like {ECO:xxx}")
print("   2. Split by separators (comma, semicolon, period)")
print("   3. Extract unique location tags for analysis")
print("\n💡 The original column preserves full annotation details")

In [ ]:
# Define the location parsing function
def parse_location(location_str):
    """
    Parse a subcellular location string from UniProt and return a list of location terms.
    
    Process:
    1. Remove curly bracket annotations like {ECO:xxx} for cleaner output
    2. Split by common separators (comma, semicolon, period)
    3. Extract unique location tags for analysis and plotting
    
    The original column can be referenced for full annotation details if needed.
    """
    if pd.isna(location_str) or location_str == '':
        return []
    
    location_str = str(location_str)
    
    # Remove curly bracket content like {ECO:0000269|PubMed:12345}
    location_str = re.sub(r'\{[^}]*\}', '', location_str)
    
    # Remove "SUBCELLULAR LOCATION:" prefix if present
    location_str = re.sub(r'^SUBCELLULAR LOCATION:\s*', '', location_str, flags=re.IGNORECASE)
    
    # Split by common separators and clean up
    # UniProt uses semicolons, commas, and periods as separators
    parts = re.split(r'[;,.]', location_str)
    
    locations = []
    for part in parts:
        # Clean up whitespace and filter empty/very short strings
        cleaned = part.strip()
        if len(cleaned) < 2:  # Skip empty or single-char remnants
            continue
        # Skip common non-location text patterns
        if cleaned.lower().startswith('note=') or cleaned.lower().startswith('note:'):
            continue
        if cleaned not in locations:
            locations.append(cleaned)
    
    return locations

print("✅ Location parsing function defined")

In [ ]:
# Demonstrate the parsing on a few examples
if 'Subcellular location [CC]' in df_raw.columns:
    print("🔬 Location parsing examples:")
    print("=" * 80)
    
    # Get a sample of unique location strings
    sample_locations = df_raw['Subcellular location [CC]'].dropna().unique()[:5]
    
    for loc in sample_locations:
        parsed = parse_location(loc)
        print(f"\nOriginal: {str(loc)[:70]}..." if len(str(loc)) > 70 else f"\nOriginal: {loc}")
        print(f"  → Parsed locations: {parsed}")
else:
    print("ℹ️ Column 'Subcellular location [CC]' not found in data.")

In [ ]:
# Apply the transformations to create a working copy
print("🔄 Applying location parsing transformations...")
print("=" * 50)

df = df_raw.copy()
print(f"Step 1: Created working copy of dataframe")
print(f"        Shape: {df.shape}")

if 'Subcellular location [CC]' in df.columns:
    # Add Location Categories column (list of all parsed locations)
    df['Location Categories'] = df['Subcellular location [CC]'].apply(parse_location)
    print(f"\nStep 2: Added 'Location Categories' column")
    print(f"        Example: {df['Location Categories'].iloc[0]}")
    
    # Show all unique locations found
    all_locs = set()
    for locs in df['Location Categories']:
        all_locs.update(locs)
    print(f"\n        All unique locations found: {sorted(all_locs)}")
else:
    print("\nSkipping location parsing - column not found")

print(f"\n✅ Final shape after transformations: {df.shape}")
print(f"   New columns added: {set(df.columns) - set(df_raw.columns)}")

In [ ]:
# View the transformed data
print("👀 Preview of transformed data (showing location columns):")
cols_to_show = ['Entry', 'Entry name']
if 'Subcellular location [CC]' in df.columns:
    cols_to_show.append('Subcellular location [CC]')
if 'Location Categories' in df.columns:
    cols_to_show.append('Location Categories')

# Filter to columns that exist
cols_to_show = [c for c in cols_to_show if c in df.columns]
df[cols_to_show].head(10)

---
## 4. Data Filtering Examples

Here we demonstrate the various filtering operations available in the dashboard.

In [ ]:
# Filter by p(LLPS) range
if 'p(LLPS)' in df.columns:
    print("🎚️ Filtering by p(LLPS) range:")
    print("=" * 50)
    
    pllps_min = df['p(LLPS)'].min()
    pllps_max = df['p(LLPS)'].max()
    print(f"Full range: {pllps_min:.3f} to {pllps_max:.3f}")
    
    # Example: Filter for high probability LLPS proteins
    threshold = 0.7
    df_high_pllps = df[df['p(LLPS)'] >= threshold]
    print(f"\nFilter: p(LLPS) >= {threshold}")
    print(f"Result: {len(df_high_pllps)} proteins (from {len(df)} total)")
    print(f"\nTop 5 high p(LLPS) proteins:")
    display_cols = ['Entry', 'Entry name', 'p(LLPS)']
    display_cols = [c for c in display_cols if c in df.columns]
    print(df_high_pllps.nlargest(5, 'p(LLPS)')[display_cols])
else:
    print("ℹ️ Column 'p(LLPS)' not found in data.")

In [ ]:
# Filter by protein length
if 'Length' in df.columns:
    print("📏 Filtering by protein length:")
    print("=" * 50)
    
    length_min = df['Length'].min()
    length_max = df['Length'].max()
    print(f"Full range: {length_min} to {length_max} amino acids")
    
    # Example: Filter for short proteins
    max_length = 300
    df_short = df[df['Length'] <= max_length]
    print(f"\nFilter: Length <= {max_length}")
    print(f"Result: {len(df_short)} proteins (from {len(df)} total)")
else:
    print("ℹ️ Column 'Length' not found in data.")

In [ ]:
# Filter by subcellular location
if 'Location Categories' in df.columns:
    print("📍 Filtering by subcellular location:")
    print("=" * 50)
    
    # Count proteins by location (proteins can have multiple locations)
    from collections import Counter
    location_counter = Counter()
    for locs in df['Location Categories']:
        location_counter.update(locs)
    
    print("Available locations and counts:")
    for loc, count in location_counter.most_common():
        print(f"  {loc}: {count}")
    
    # Example: Filter for nuclear proteins
    location_filter = 'Nucleus'
    df_nuclear = df[df['Location Categories'].apply(lambda x: location_filter in x)]
    print(f"\nFilter: '{location_filter}' in Location Categories")
    print(f"Result: {len(df_nuclear)} proteins")
    
    if 'p(LLPS)' in df.columns and len(df_nuclear) > 0:
        print(f"Mean p(LLPS) for nuclear proteins: {df_nuclear['p(LLPS)'].mean():.3f}")
else:
    print("ℹ️ Location columns not available.")

In [ ]:
# Combined filtering
print("🔗 Combined filtering example:")
print("=" * 50)

df_filtered = df.copy()
filters_applied = []

# Apply p(LLPS) filter
if 'p(LLPS)' in df.columns:
    pllps_threshold = 0.5
    df_filtered = df_filtered[df_filtered['p(LLPS)'] >= pllps_threshold]
    filters_applied.append(f"p(LLPS) >= {pllps_threshold}")

# Apply length filter
if 'Length' in df.columns:
    min_length, max_length = 100, 1000
    df_filtered = df_filtered[(df_filtered['Length'] >= min_length) & 
                               (df_filtered['Length'] <= max_length)]
    filters_applied.append(f"Length between {min_length} and {max_length}")

print("Filters applied:")
for f in filters_applied:
    print(f"  • {f}")

print(f"\nOriginal count: {len(df)}")
print(f"Filtered count: {len(df_filtered)}")
print(f"Removed: {len(df) - len(df_filtered)} proteins")

In [ ]:
# Text search example
print("🔍 Text search example:")
print("=" * 50)

search_term = "kinase"  # Example search term
search_columns = [col for col in ['Entry', 'Entry name', 'Protein names'] if col in df.columns]

if search_columns:
    mask = pd.Series([False] * len(df))
    for col in search_columns:
        mask = mask | df[col].astype(str).str.contains(search_term, case=False, na=False)
    
    df_search = df[mask]
    print(f"Search term: '{search_term}'")
    print(f"Searched in columns: {search_columns}")
    print(f"\nFound {len(df_search)} matching proteins:")
    
    if len(df_search) > 0:
        print(df_search[search_columns].head(10))
else:
    print("No searchable columns found.")

---
## 5. Statistical Analysis

Detailed statistical analysis of the dataset.

In [ ]:
# Summary statistics by location
if 'Location Categories' in df.columns and 'p(LLPS)' in df.columns:
    print("📊 Statistics by Subcellular Location:")
    print("=" * 70)
    
    # Explode location categories for analysis
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    
    location_stats = df_exploded.groupby('Location Categories').agg({
        'Entry': 'count',
        'p(LLPS)': ['mean', 'std', 'min', 'max']
    }).round(3)
    
    location_stats.columns = ['Count', 'Mean p(LLPS)', 'Std p(LLPS)', 'Min p(LLPS)', 'Max p(LLPS)']
    location_stats = location_stats.sort_values('Count', ascending=False)
    
    print(location_stats)
    print("\n💡 Note: Proteins with multiple locations are counted in each location")
else:
    print("ℹ️ Required columns for location statistics not available.")

In [ ]:
# Correlation analysis
print("📈 Correlation Analysis:")
print("=" * 50)

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

if len(numeric_cols) >= 2:
    corr_matrix = df[numeric_cols].corr()
    print("\nCorrelation matrix:")
    print(corr_matrix.round(3))
    
    # Highlight specific correlations
    if 'Length' in numeric_cols and 'p(LLPS)' in numeric_cols:
        corr_length_pllps = df['Length'].corr(df['p(LLPS)'])
        print(f"\n🔗 Length vs p(LLPS) correlation: {corr_length_pllps:.4f}")
else:
    print("Not enough numeric columns for correlation analysis.")

In [ ]:
# Distribution percentiles
if 'p(LLPS)' in df.columns:
    print("📊 p(LLPS) Distribution Percentiles:")
    print("=" * 50)
    
    percentiles = [10, 25, 50, 75, 90, 95, 99]
    for p in percentiles:
        value = df['p(LLPS)'].quantile(p/100)
        count_above = (df['p(LLPS)'] >= value).sum()
        print(f"  {p:2}th percentile: {value:.3f} ({count_above} proteins at or above)")

---
## 6. Visualizations

Interactive visualizations to explore the data.

In [ ]:
# p(LLPS) distribution histogram
if 'p(LLPS)' in df.columns:
    print("📊 Distribution of p(LLPS):")
    
    fig = px.histogram(
        df, 
        x='p(LLPS)',
        nbins=50,
        title="Distribution of LLPS Probability",
        labels={'p(LLPS)': 'Probability of LLPS'},
        color_discrete_sequence=['#1f77b4']
    )
    fig.update_layout(
        xaxis_title="p(LLPS)",
        yaxis_title="Count",
        showlegend=False
    )
    fig.show()
else:
    print("ℹ️ Column 'p(LLPS)' not found.")

In [ ]:
# Protein length distribution
if 'Length' in df.columns:
    print("📏 Distribution of Protein Lengths:")
    
    fig = px.histogram(
        df,
        x='Length',
        nbins=50,
        title="Distribution of Protein Lengths",
        labels={'Length': 'Protein Length (amino acids)'},
        color_discrete_sequence=['#2ecc71']
    )
    fig.update_layout(
        xaxis_title="Length (amino acids)",
        yaxis_title="Count"
    )
    fig.show()
else:
    print("ℹ️ Column 'Length' not found.")

In [ ]:
# Scatter plot: Length vs p(LLPS)
if 'Length' in df.columns and 'p(LLPS)' in df.columns:
    print("📈 Scatter Plot: Length vs p(LLPS)")
    
    # Color by location if available
    color_col = None
    
    hover_data = [col for col in ['Entry', 'Entry name'] if col in df.columns]
    
    fig = px.scatter(
        df,
        x='Length',
        y='p(LLPS)',
        color=color_col,
        hover_data=hover_data if hover_data else None,
        title="p(LLPS) vs Protein Length"
    )
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.update_layout(
        xaxis_title="Protein Length (amino acids)",
        yaxis_title="p(LLPS)"
    )
    fig.show()
else:
    print("ℹ️ Required columns for scatter plot not found.")

In [ ]:
# Location distribution bar chart
if 'Location Categories' in df.columns:
    print("📍 Protein Count by Subcellular Location:")
    
    # Explode and count locations
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    location_counts = df_exploded['Location Categories'].value_counts()
    
    fig = px.bar(
        x=location_counts.index,
        y=location_counts.values,
        title="Protein Count by Subcellular Location (proteins may appear in multiple)",
        labels={'x': 'Subcellular Location', 'y': 'Number of Proteins'},
        color=location_counts.index,
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, xaxis_tickangle=-45)
    fig.show()
else:
    print("ℹ️ Location column not available.")

In [ ]:
# Box plot: p(LLPS) by location
if 'Location Categories' in df.columns and 'p(LLPS)' in df.columns:
    print("📊 p(LLPS) Distribution by Subcellular Location:")
    
    # Explode locations for box plot
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    
    fig = px.box(
        df_exploded,
        x='Location Categories',
        y='p(LLPS)',
        title="p(LLPS) Distribution by Subcellular Location",
        color='Location Categories',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, xaxis_tickangle=-45)
    fig.show()
else:
    print("ℹ️ Required columns for box plot not available.")

---
## 📝 Notes

This notebook mirrors the data transformations in the Streamlit dashboard (`app.py`) but allows you to:

1. **See intermediate steps**: Each transformation is broken down so you can inspect the data before and after
2. **Modify parameters**: Easily change filter thresholds, search terms, etc.
3. **Add custom analysis**: Extend with your own code cells
4. **Export results**: Save filtered datasets or figures as needed

### Exporting Filtered Data

In [ ]:
# Export filtered data to CSV
# Uncomment and modify the line below to export

# df_filtered.to_csv('filtered_proteins.csv', index=False)
# print("✅ Data exported to filtered_proteins.csv")

---
*Generated from the LLPS Protein Data Explorer project*